# obj4 · Rácios de consórcio e perfil de secreção (MICOM)

**TASK 3.1** — comunidades microbianas para vinho de baixo teor alcoólico

Que composição de consórcio dá o menor rendimento em etanol sem a comunidade deixar de crescer,
e para onde vai o carbono que não foi para etanol. A saída é a escolha dos consórcios que passam ao dFBA.

**A regra deste notebook: não se edita nada.** Carregam-se os ficheiros curados do `obj3`, aplica-se o
MS300 como está definido no `obj2`, simula-se e reporta-se. Não se renomeiam reações, não se libertam
bounds, não se acrescentam suplementos fora da lista declarada no `obj2`, e não se substitui nenhuma
função do MICOM. Se um resultado exigir mais alguma coisa, isso decide-se **depois** de ver o resultado
e entra num notebook seguinte, declarado.

| código | espécie | ficheiro | regime |
|---|---|---|---|
| `Sc` | *Saccharomyces cerevisiae* | `Scer_smetana.xml` | anaerobiose |
| `Td` | *Torulaspora delbrueckii* | `Tdel_smetana.xml` | anaerobiose |
| `Mp` | *Metschnikowia pulcherrima* | `Mpul_smetana.xml` | microaerobiose, q(O₂) = 0.5 |
| `Lt` | *Lachancea thermotolerans* | `Ltho_smetana.xml` | anaerobiose |

## 1 · Preparação

In [1]:
import os, sys, itertools, warnings
from pathlib import Path
import pandas as pd
import cobra, micom
from micom import Community

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

def raiz(p=None):
    p = (Path(p) if p else Path.cwd()).resolve()
    for c in (p, *p.parents):
        if (c / "models").is_dir() and (c / "scripts").is_dir():
            return c
    raise RuntimeError("raiz do projecto nao encontrada")

ROOT = raiz()
sys.path.insert(0, str(ROOT / "scripts"))
import meio_mosto as MM
import consorcios as CM

OUT = ROOT / "results" / "obj4"
OUT.mkdir(parents=True, exist_ok=True)

_lic = os.path.expanduser("~/gurobi.lic")
if os.path.exists(_lic):
    os.environ["GRB_LICENSE_FILE"] = _lic
SOLVER = "gurobi"

print("cobra", cobra.__version__, "| micom", micom.__version__)
print("raiz:", ROOT)

cobra 0.32.1 | micom 0.39.1
raiz: /Users/Sara_1/Desktop/WORK/TASK3.1


In [2]:
# Os ficheiros do obj3: espaco de nomes extracelular reconciliado e regime de O2
# ja aplicado por organismo. NAO se usam os *_curado.xml, que sao a variante aerobia.
HARM = ROOT / "models" / "harmonizados"
FICHEIRO = {"Sc": HARM / "Scer_smetana.xml",
            "Td": HARM / "Tdel_smetana.xml",
            "Mp": HARM / "Mpul_smetana.xml",
            "Lt": HARM / "Ltho_smetana.xml"}

ESPECIE = {"Sc": "Saccharomyces cerevisiae", "Td": "Torulaspora delbrueckii",
           "Mp": "Metschnikowia pulcherrima", "Lt": "Lachancea thermotolerans"}
BIOMASSA = {"Sc": "r_2111", "Td": "BIOMASS", "Mp": "BIOMASS", "Lt": "BIOMASS"}

# Referencias medidas no obj2 (MS300, cenario central, regime proprio).
MU_OBJ2 = {"Sc": 0.1654, "Td": 0.1835, "Mp": 0.1836, "Lt": 0.1835}
Q_O2 = dict(CM.Q_O2)

# Lista declarada na seccao 5.1 do obj2, com a justificacao escrita la.
# Nada entra no meio para alem disto.
SUPLEMENTOS = MM.SUPLEMENTOS

FRACTION = 0.5                      # cooperative_tradeoff

# Piso de crescimento imposto a CADA membro, como fraccao do seu mu solo.
# Sem isto o MICOM maximiza a SOMA das taxas, e uma soma e maxima dando o
# substrato todo ao membro mais eficiente: o outro sai a zero. Nao e biologia,
# e a forma da pergunta.
#
# Medido no par Sc+Td a 50/50, com estes ficheiros e este meio:
#     piso     mu_Sc     mu_Td    mu_comunidade
#     nenhum   0.0000    0.4000   0.20002
#     10 %     0.0165    0.3806   0.19858
#     50 %     0.0827    0.3030   0.19284
#     90 %     0.1489    0.2253   0.18709
#     100 %    0.1654    0.2059   0.18566
# Obrigar as duas ao seu mu solo custa 7 % do crescimento da comunidade. O
# criterio diz-se numa frase: cada levedura tem de crescer pelo menos tao
# depressa como cresce sozinha no mesmo mosto.
MIN_GROWTH_FRAC = 1.00

NIVEIS = (0.95, 0.90, 0.75)         # dominancia de um membro, mais partes iguais

def piso(membros):
    return {c: MIN_GROWTH_FRAC * MU_OBJ2[c] for c in membros}

print("suplementos declarados:", SUPLEMENTOS)
print("O2 por organismo       :", Q_O2)
print("piso por membro        :", {c: round(v, 4) for c, v in piso(MU_OBJ2).items()})

# O MICOM precisa de um solver QUADRATICO para distribuir o crescimento pelos
# membros. Sem ele troca quadrados por valores absolutos, e nessa metrica a
# solucao "dar tudo ao mais eficiente" passa a ganhar.
import optlang
_qp = {k for k, v in optlang.available_solvers.items() if v} & {"GUROBI", "CPLEX", "OSQP"}
print("solvers QP disponiveis :", sorted(_qp) or "NENHUM")
if not _qp:
    print("  AVISO: sem solver QP o MICOM nao distribui o crescimento pelos membros.")

suplementos declarados: ('iron', 'palmitoleate', 'ergosterol', 'oleate', 'nicotinate')
O2 por organismo       : {'Sc': 0.0, 'Td': 0.0, 'Mp': 0.5, 'Lt': 0.0}
piso por membro        : {'Sc': 0.1654, 'Td': 0.1835, 'Mp': 0.1836, 'Lt': 0.1835}
solvers QP disponiveis : ['GUROBI', 'OSQP']


In [3]:
M = {cod: cobra.io.read_sbml_model(str(p)) for cod, p in FICHEIRO.items()}

linhas = []
for cod, m in M.items():
    obj = [r.id for r in m.reactions if r.objective_coefficient != 0]
    linhas.append({"codigo": cod, "especie": ESPECIE[cod], "ficheiro": FICHEIRO[cod].name,
                   "reaccoes": len(m.reactions), "trocas": len(m.exchanges),
                   "objectivo": ", ".join(obj) or "(nenhum)"})
d_mod = pd.DataFrame(linhas)
print(d_mod.to_string(index=False))

# Controlo de atribuicao de ficheiros. As quatro contagens sao distintas e
# ordenam-se Td < Mp < Lt < Sc; se a ordem falhar, um ficheiro esta trocado.
n = dict(zip(d_mod.codigo, d_mod.reaccoes))
if not (n["Td"] < n["Mp"] < n["Lt"] < n["Sc"]):
    raise RuntimeError(f"contagens fora da ordem esperada: {n}. Verificar FICHEIRO.")

for cod, m in M.items():
    if BIOMASSA[cod] not in m.reactions:
        raise RuntimeError(f"{cod}: reaccao de biomassa {BIOMASSA[cod]} ausente")
    m.objective = BIOMASSA[cod]

print("\nficheiros e objectivos verificados")

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
codigo                   especie         ficheiro  reaccoes  trocas objectivo
    Sc  Saccharomyces cerevisiae Scer_smetana.xml      4102     270    r_2111
    Td   Torulaspora delbrueckii Tdel_smetana.xml      1971     240   BIOMASS
    Mp Metschnikowia pulcherrima Mpul_smetana.xml      2131     254   BIOMASS
    Lt  Lachancea thermotolerans Ltho_smetana.xml      2139     252   BIOMASS

ficheiros e objectivos verificados


## 2 · Teste de aceitação

Cada modelo sozinho, com o MS300 aplicado à maneira do `obj2` e no seu próprio regime, tem de devolver
o µ solo canónico. Se não devolver, tudo o que vem a seguir é sobre modelos diferentes dos que foram
curados, e o notebook pára aqui.

In [4]:
linhas = []
for cod, m in M.items():
    with m:
        MM.aplicar_meio_mosto(m, q_o2=Q_O2[cod], suplementos=SUPLEMENTOS, verbose=False)
        mu = m.slim_optimize()
    mu = 0.0 if mu != mu else mu
    linhas.append({"codigo": cod, "especie": ESPECIE[cod], "q_O2": Q_O2[cod],
                   "mu": round(mu, 4), "mu_obj2": MU_OBJ2[cod],
                   "delta": round(mu - MU_OBJ2[cod], 6)})

d_aceit = pd.DataFrame(linhas)
print(d_aceit.to_string(index=False))
d_aceit.to_csv(OUT / "aceitacao_solo.csv", index=False)

if d_aceit.delta.abs().max() > 1e-3:
    raise RuntimeError("os modelos nao reproduzem o mu do obj2. Parar aqui.")
print("\nteste de aceitacao passou")

codigo                   especie  q_O2     mu  mu_obj2    delta
    Sc  Saccharomyces cerevisiae   0.0 0.1654   0.1654 0.000024
    Td   Torulaspora delbrueckii   0.0 0.1835   0.1835 0.000033
    Mp Metschnikowia pulcherrima   0.5 0.1836   0.1836 0.000000
    Lt  Lachancea thermotolerans   0.0 0.1835   0.1835 0.000033

teste de aceitacao passou


## 3 · Meio da comunidade

Num modelo isolado quem define o meio são os bounds das trocas do próprio organismo. Numa comunidade
passa a ser o **pool partilhado**: as trocas `_m` do modelo fundido. As trocas `_e` de cada organismo
ficam abertas, porque é por elas que um capta o que o outro secretou.

O oxigénio entra só pelo pool. As trocas internas de O₂ dos três modelos anaeróbios já vêm a zero nos
ficheiros do `obj3`, por isso abrir o pool não lhes dá oxigénio nenhum; só a *M. pulcherrima* o pode captar.

**Não se usa `com.medium = ...`.** No MICOM 0.39 esse setter fecha também as trocas `_e` ausentes do
dicionário, o que corta a ligação de cada organismo ao pool e anula qualquer alimentação cruzada.
Escrevem-se os bounds diretamente nas trocas `_m`.

In [5]:
taxonomia = pd.DataFrame([
    {"id": cod, "species": ESPECIE[cod], "abundance": 1.0 / len(M),
     "file": str(FICHEIRO[cod])}
    for cod in M
])

com_ref = Community(taxonomia, solver=SOLVER, progress=False)
EX_M = {r.id for r in com_ref.exchanges if r.id.endswith("_m")}
print("organismos          :", com_ref.taxa)
print("trocas da comunidade:", len(com_ref.exchanges), f"({len(EX_M)} do pool)")

organismos          : ['Sc', 'Td', 'Mp', 'Lt']
trocas da comunidade: 390 (390 do pool)


In [6]:
mapa_met, divergencias = CM.mapa_especie_metabolito(M)
if divergencias:
    print("DIVERGENCIAS entre modelos para o mesmo composto:", divergencias)

def troca_do_pool(met_id):
    """`glcD_e` -> `EX_glcD_m`, que e como o MICOM nomeia a troca do pool."""
    stem = met_id[:-2] if met_id.endswith("_e") else met_id
    return f"EX_{stem}_m"

comp = MM.composicao_mmol_L()
meio, fora = {}, []

# 1. a receita, pela equacao do obj2
for especie, c_mmol_L in comp.items():
    met = mapa_met.get(especie)
    ex = troca_do_pool(met) if met else None
    if ex is None or ex not in EX_M:
        fora.append(especie)
        continue
    meio[ex] = abs(MM.calcular_bound(c_mmol_L))

# 2. basais sem tecto: solvente, pH e produto, nao sao nutrientes
for especie in ("water", "proton", "co2"):
    met = mapa_met.get(especie)
    ex = troca_do_pool(met) if met else None
    if ex in EX_M:
        meio[ex] = 1000.0

# 3. `ash 1 g` dos CarveFungi, derivado da receita (1.867 g/L de sais), como no obj2
met_ash = mapa_met.get("ash")
if met_ash and troca_do_pool(met_ash) in EX_M:
    meio[troca_do_pool(met_ash)] = abs(MM.calcular_bound(MM.ash_mmol_L()))

# 4. suplementos declarados no obj2 5.1, com o mesmo tecto
for especie in SUPLEMENTOS:
    met = mapa_met.get(especie)
    ex = troca_do_pool(met) if met else None
    if ex in EX_M:
        meio[ex] = max(meio.get(ex, 0.0), MM.Q_SUPLEMENTO)
    else:
        fora.append(especie)

# 5. oxigenio no pool, ao valor do unico organismo que o pode captar
met_o2 = mapa_met.get("oxygen")
if met_o2 and troca_do_pool(met_o2) in EX_M:
    meio[troca_do_pool(met_o2)] = max(Q_O2.values())

print(f"{len(meio)} trocas do pool abertas")
print(f"sem troca no pool ({len(fora)}): {sorted(set(fora))}")
pd.Series(meio, name="bound").sort_values().to_csv(OUT / "meio_comunidade.csv")

48 trocas do pool abertas
sem troca no pool (4): ['borate', 'cobalt', 'iodide', 'molybdate']


In [7]:
def aplicar_meio_comunidade(com, meio):
    """Escreve os bounds nas trocas `_m` do pool. Nao toca nas trocas `_e`."""
    abertas = 0
    for r in com.exchanges:
        if not r.id.endswith("_m"):
            continue
        r.lower_bound = -float(meio.get(r.id, 0.0))
        abertas += r.id in meio
    return abertas

def o2_interno(com):
    """Bound de captacao de O2 de cada organismo, como veio do ficheiro."""
    fora = {}
    for r in com.exchanges:
        if r.id.endswith("_m") or "o2" not in r.id.lower():
            continue
        fora[r.id] = round(r.lower_bound, 3)
    return fora

aplicar_meio_comunidade(com_ref, meio)
print("O2 por organismo, como veio dos ficheiros:")
for rid, lb in sorted(o2_interno(com_ref).items()):
    print(f"   {rid:24s} lb = {lb}")

O2 por organismo, como veio dos ficheiros:


## 4 · Controlo: os membros conseguem crescer ao mesmo tempo?

Antes do varrimento, a pergunta que interessa sobre um par não é o que o ótimo escolhe, é o que é
**possível**. Força-se o mesmo piso de crescimento nos dois e vê-se até onde continua viável.

A distinção importa e foi o que fez a primeira versão deste notebook parar. O ótimo do MICOM maximiza
a **soma** das taxas de crescimento, e uma soma fica maior dando o substrato todo ao membro que o
converte melhor, o que deixa o outro a zero sem que nada no meio o impeça. A viabilidade simultânea é
a propriedade da comunidade; o que o ótimo escolhe é outra coisa.

Por isso, a partir daqui todas as simulações levam um **piso de crescimento por membro**
(`MIN_GROWTH_FRAC`), que é o parâmetro `min_growth` do próprio MICOM. Só entram nos resultados as
configurações em que todos crescem.

In [8]:
HEXOSES = ["glucose", "fructose"]

def _ex_pool(especie):
    """Troca do pool a partir do nome da especie na receita do MS300."""
    met = mapa_met.get(especie)
    return troca_do_pool(met) if met else None

def _ex_stem(stem):
    """Troca do pool a partir do identificador BiGG, para produtos que nao
    constam da receita e por isso nao estao no mapa da composicao."""
    ex = f"EX_{stem}_m"
    return ex if ex in EX_M else None

EX_HEXOSE = [_ex_pool(k) for k in HEXOSES if _ex_pool(k)]
EX_ETANOL = _ex_stem("etoh")
if not EX_HEXOSE or EX_ETANOL is None:
    raise RuntimeError(f"trocas em falta no pool: hexose={EX_HEXOSE}, etanol={EX_ETANOL}")

def resumo_carbono(sol):
    """Le do pool (linha `medium`): o que entra, o que sai e o rendimento."""
    if "medium" not in sol.fluxes.index:
        return {}
    f = sol.fluxes.loc["medium"]
    hexo = -sum(float(f.get(ex, 0.0)) for ex in EX_HEXOSE)
    etoh = float(f.get(EX_ETANOL, 0.0))
    return {"hexose": round(hexo, 4), "etanol": round(etoh, 4),
            "Y_mol_mol": round(etoh / hexo, 4) if hexo > 1e-9 else None,
            "Y_g_g": round(etoh * 46.07 / (hexo * 180.16), 4) if hexo > 1e-9 else None}

PAR = ["Sc", "Td"]
com_t = Community(taxonomia[taxonomia.id.isin(PAR)], solver=SOLVER, progress=False)
com_t.set_abundance({c: 0.5 for c in PAR}, normalize=True)
aplicar_meio_comunidade(com_t, meio)

def bio_na_comunidade(cod):
    """No modelo fundido a biomassa de cada organismo chama-se `<id>__<cod>`."""
    return f"{BIOMASSA[cod]}__{cod}"

print("cada membro sozinho dentro da comunidade:")
for cod in PAR:
    with com_t:
        com_t.objective = bio_na_comunidade(cod)
        v = com_t.slim_optimize()
    print(f"   {cod}: {0.0 if v != v else v:.5f}")

print("\nate onde e viavel um piso IGUAL nos dois:")
maior_viavel = 0.0
for alvo in (0.02, 0.05, 0.10, 0.15, 0.20, 0.25):
    with com_t:
        for cod in PAR:
            com_t.reactions.get_by_id(bio_na_comunidade(cod)).lower_bound = alvo
        v = com_t.slim_optimize()
    viavel = v == v and v is not None
    maior_viavel = alvo if viavel else maior_viavel
    print(f"   ambos >= {alvo:.3f}: {'viavel' if viavel else 'inviavel'}")

print(f"   (a escada e informativa: o limite real cai ENTRE dois degraus)")

# A decisao testa o piso que vai mesmo ser usado, por organismo, e nao a escada.
with com_t:
    for cod, lb in piso(PAR).items():
        com_t.reactions.get_by_id(bio_na_comunidade(cod)).lower_bound = lb
    v = com_t.slim_optimize()
if not (v == v and v is not None):
    raise RuntimeError(f"o piso {piso(PAR)} nao e viavel neste par. Verificar o meio.")
print(f"\ncontrolo passou: o piso {({k: round(x,4) for k, x in piso(PAR).items()})} e viavel")

print("\ncom o piso aplicado, o que o cooperative_tradeoff devolve:")
sol_t = com_t.cooperative_tradeoff(min_growth=piso(PAR), fraction=FRACTION, fluxes=True)
print("   status:", sol_t.status, "| mu comunidade:", round(sol_t.growth_rate, 5))
print(sol_t.members[["abundance", "growth_rate"]].to_string())
print("   carbono no pool:", resumo_carbono(sol_t))

cada membro sozinho dentro da comunidade:
   Sc: 0.34085
   Td: 0.40004

ate onde e viavel um piso IGUAL nos dois:
   ambos >= 0.020: viavel
   ambos >= 0.050: viavel
   ambos >= 0.100: viavel
   ambos >= 0.150: viavel
   ambos >= 0.200: inviavel
   ambos >= 0.250: inviavel
   (a escada e informativa: o limite real cai ENTRE dois degraus)

controlo passou: o piso {'Sc': 0.1654, 'Td': 0.1835} e viavel

com o piso aplicado, o que o cooperative_tradeoff devolve:
   status: optimal | mu comunidade: 0.18404
              abundance  growth_rate
compartments                        
Sc                  0.5     0.184588
Td                  0.5     0.183499
medium              NaN          NaN
   carbono no pool: {'hexose': 38.2798, 'etanol': 57.7553, 'Y_mol_mol': 1.5088, 'Y_g_g': 0.3858}


## 5 · Varrimento de rácios

Todos os consórcios (6 pares, 4 trios, o quarteto) × os rácios da grelha. Cada rácio é partes iguais
mais cada membro dominante a 95, 90 e 75 %.

O nível de 95 % está lá de propósito: o piso de 10 % por organismo que a revisão anterior usava tira
o regime de baixa inoculação, e é esse que o ensaio real vai fazer.

Os fluxos brutos são guardados em CSV **antes** de qualquer agregação, para as secções seguintes
poderem mudar de critério sem voltar a correr as simulações.

In [9]:
def racios(n, niveis=NIVEIS):
    """Partes iguais, mais cada membro dominante a cada nivel."""
    out = [tuple([1.0 / n] * n)]
    for d in niveis:
        if d <= 1.0 / n:
            continue
        resto = (1.0 - d) / (n - 1)
        out += [tuple(d if j == i else resto for j in range(n)) for i in range(n)]
    return out

CODIGOS = ["Sc", "Td", "Mp", "Lt"]
CONSORCIOS = {"+".join(c): list(c)
              for k in (2, 3, 4)
              for c in itertools.combinations(CODIGOS, k)}

total = sum(len(racios(len(v))) for v in CONSORCIOS.values())
print(f"{len(CONSORCIOS)} consorcios, {total} pontos")

11 consorcios, 95 pontos


In [10]:
import time

registos, fluxos = [], []
t0 = time.time()

for nome, membros in CONSORCIOS.items():
    tax = taxonomia[taxonomia.id.isin(membros)].copy()
    for r in racios(len(membros)):
        etiqueta = "/".join(f"{x:.2f}" for x in r)
        com = Community(tax, solver=SOLVER, progress=False)
        com.set_abundance(dict(zip(membros, r)), normalize=True)
        aplicar_meio_comunidade(com, meio)

        linha = {"consorcio": nome, "n": len(membros), "racio": etiqueta}
        linha.update({f"X_{c}": round(x, 4) for c, x in zip(membros, r)})
        try:
            # min_growth: cada membro tem de crescer. Um racio onde isso seja
            # impossivel devolve infactivel, que e o resultado a registar.
            sol = com.cooperative_tradeoff(min_growth=piso(membros),
                                           fraction=FRACTION, fluxes=True)
        except Exception as exc:
            linha["status"] = f"inviavel/erro: {type(exc).__name__}"
            registos.append(linha)
            continue

        linha["status"] = sol.status
        linha["mu_comunidade"] = round(sol.growth_rate, 5)
        for c in membros:
            linha[f"mu_{c}"] = round(float(sol.members.loc[c, "growth_rate"]), 5)
        linha.update(resumo_carbono(sol))
        registos.append(linha)

        f = sol.fluxes.copy()
        f.insert(0, "racio", etiqueta)
        f.insert(0, "consorcio", nome)
        f.index.name = "organismo"
        fluxos.append(f.reset_index())

d_var = pd.DataFrame(registos)
d_var.to_csv(OUT / "varrimento_racios.csv", index=False)
pd.concat(fluxos, ignore_index=True).to_csv(OUT / "varrimento_fluxos.csv", index=False)

n_ok = (d_var.status == "optimal").sum()
print(f"[{time.time()-t0:.0f}s]  {n_ok}/{len(d_var)} com TODOS os membros a crescer")
if (d_var.status != "optimal").any():
    print("\nracios onde nao e possivel todos crescerem (resultado, nao erro):")
    print(d_var[d_var.status != "optimal"][["consorcio", "racio", "status"]].to_string(index=False))

# Controlo: nenhum rendimento acima do maximo teorico de 0.511 g/g
acima = d_var[d_var.Y_g_g.notna() & (d_var.Y_g_g > 0.511)]
if len(acima):
    print(acima.to_string(index=False))
    raise RuntimeError(f"{len(acima)} pontos acima do maximo teorico. Parar aqui.")
print("nenhum rendimento acima de 0.511 g/g")

[1177s]  56/95 com TODOS os membros a crescer

racios onde nao e possivel todos crescerem (resultado, nao erro):
  consorcio               racio                     status
      Sc+Td           0.95/0.05 inviavel/erro: GurobiError
      Sc+Td           0.25/0.75 inviavel/erro: GurobiError
      Sc+Mp           0.95/0.05 inviavel/erro: GurobiError
      Sc+Mp           0.05/0.95 inviavel/erro: GurobiError
      Sc+Mp           0.10/0.90 inviavel/erro: GurobiError
      Sc+Mp           0.25/0.75 inviavel/erro: GurobiError
      Sc+Lt           0.05/0.95 inviavel/erro: GurobiError
      Td+Mp           0.95/0.05 inviavel/erro: GurobiError
      Td+Mp           0.90/0.10 inviavel/erro: GurobiError
      Td+Mp           0.75/0.25 inviavel/erro: GurobiError
      Mp+Lt           0.95/0.05 inviavel/erro: GurobiError
      Mp+Lt           0.05/0.95 inviavel/erro: GurobiError
      Mp+Lt           0.10/0.90 inviavel/erro: GurobiError
      Mp+Lt           0.25/0.75 inviavel/erro: GurobiError
  

In [11]:
ok = d_var[d_var.status == "optimal"].copy()
print("menor rendimento em etanol por consorcio\n")
melhor = (ok.sort_values("Y_g_g")
            .groupby("consorcio", as_index=False).first()
            .sort_values("Y_g_g"))
print(melhor[["consorcio", "racio", "mu_comunidade", "hexose",
              "etanol", "Y_mol_mol", "Y_g_g"]].to_string(index=False))
melhor.to_csv(OUT / "melhor_por_consorcio.csv", index=False)

menor rendimento em etanol por consorcio

  consorcio               racio  mu_comunidade  hexose  etanol  Y_mol_mol  Y_g_g
      Sc+Td           0.75/0.25        0.17725 38.2800 47.3845     1.2378 0.3165
      Sc+Lt           0.25/0.75        0.19083 38.2785 57.6215     1.5053 0.3849
   Sc+Mp+Lt      0.33/0.33/0.33        0.18856 38.2791 58.3732     1.5249 0.3900
   Sc+Td+Lt      0.33/0.33/0.33        0.18857 38.2789 59.0774     1.5433 0.3947
      Sc+Mp           0.50/0.50        0.18404 38.2797 59.5088     1.5546 0.3975
Sc+Td+Mp+Lt 0.08/0.08/0.75/0.08        0.19761 38.2770 59.8902     1.5647 0.4001
   Sc+Td+Mp      0.33/0.33/0.33        0.18856 38.2785 60.1669     1.5718 0.4019
      Td+Lt           0.10/0.90        0.18350 38.1446 62.1877     1.6303 0.4169
      Td+Mp           0.25/0.75        0.18357 37.5460 62.6111     1.6676 0.4264
   Td+Mp+Lt      0.12/0.12/0.75        0.18351 37.7236 63.5642     1.6850 0.4309
      Mp+Lt           0.75/0.25        0.18357 38.0572 64.8222     

## 6 · Perfil de secreção

O que sai de cada comunidade, lido dos fluxos já guardados. A linha `medium` é o balanço líquido do
pool, ou seja o que a comunidade como um todo liberta ou consome.

**Dois eixos que não são avaliáveis aqui, e têm de ir na legenda da figura e não só no texto:**

- A *M. pulcherrima* **não produz glicerol em cenário nenhum**, porque falta a glicerol-3-fosfatase no
  ficheiro (`UF01626_C` existe na Td e na Lt). Qualquer consórcio com ela aparece artificialmente baixo
  nesse eixo.
- A **via de Ehrlich não existe nos três CarveFungi**. Dos 11 compostos da via, só 4 têm pool comum, e
  os quatro produtos finais com peso sensorial (2-feniletanol, triptofol, tirosol, 3-metilbutanal) só
  existem no yeast-GEM. O aroma não é avaliável aqui.

Se um produto sair zero em todas as configurações, **reporta zero e explica porquê**. Não se acrescenta
manutenção nem tectos para o forçar: isso seria escolher o resultado.

In [12]:
# Identificadores BiGG, como o obj3 os deixou depois da harmonizacao.
PRODUTOS = {"etanol": "etoh", "glicerol": "glyc", "L-lactato": "lacL",
            "D-lactato": "lacD", "acetato": "ac", "succinato": "succ",
            "acetaldeido": "acald", "CO2": "co2", "piruvato": "pyr"}

ALVOS, sem_troca = {}, []
for nome, stem in PRODUTOS.items():
    ex = _ex_stem(stem)
    if ex:
        ALVOS[nome] = ex
    else:
        sem_troca.append(f"{nome} (EX_{stem}_m)")
print("seguidos:", list(ALVOS))
if sem_troca:
    print("sem troca no pool (declarar):", sem_troca)

seguidos: ['etanol', 'glicerol', 'L-lactato', 'D-lactato', 'acetato', 'succinato', 'acetaldeido', 'CO2', 'piruvato']


In [13]:
d_fx = pd.read_csv(OUT / "varrimento_fluxos.csv")
pool = d_fx[d_fx.organismo == "medium"]

linhas = []
for (nome, etiqueta), g in pool.groupby(["consorcio", "racio"]):
    linha = {"consorcio": nome, "racio": etiqueta}
    hexo = -sum(float(g[ex].iloc[0]) for ex in EX_HEXOSE if ex in g.columns)
    linha["hexose"] = round(hexo, 4)
    for prod, ex in ALVOS.items():
        v = float(g[ex].iloc[0]) if ex in g.columns else 0.0
        linha[prod] = round(v, 4)
        linha[f"Y_{prod}"] = round(v / hexo, 4) if hexo > 1e-9 else None
    linhas.append(linha)

d_sec = pd.DataFrame(linhas)
d_sec.to_csv(OUT / "secrecao_comunidade.csv", index=False)

# O ponto de menor etanol de cada consorcio, que e o que se compara
chave = melhor[["consorcio", "racio"]]
print(chave.merge(d_sec, on=["consorcio", "racio"]).to_string(index=False))

nulos = [p for p in ALVOS if d_sec[p].abs().max() < 1e-6]
print(f"\nprodutos com fluxo nulo em TODAS as configuracoes: {nulos or 'nenhum'}")

  consorcio               racio  hexose  etanol  Y_etanol  glicerol  Y_glicerol  L-lactato  Y_L-lactato  D-lactato  Y_D-lactato  acetato  Y_acetato  succinato  Y_succinato  acetaldeido  Y_acetaldeido     CO2  Y_CO2  piruvato  Y_piruvato
      Sc+Td           0.75/0.25 38.2800 47.3845    1.2378    0.0002      0.0000     0.0000       0.0000        0.0          0.0   0.0017     0.0000    16.8722       0.4408       0.2861         0.0075 50.5623 1.3209    0.0006      0.0000
      Sc+Lt           0.25/0.75 38.2785 57.6215    1.5053    0.8288      0.0217     0.0018       0.0000        0.0          0.0   0.2150     0.0056     9.3545       0.2444       0.0024         0.0001 62.7705 1.6398    0.0017      0.0000
   Sc+Mp+Lt      0.33/0.33/0.33 38.2791 58.3732    1.5249    1.6630      0.0434     0.1532       0.0040        0.0          0.0   0.0018     0.0000     5.9684       0.1559       0.0002         0.0000 66.9451 1.7489    0.0214      0.0006
   Sc+Td+Lt      0.33/0.33/0.33 38.2789 59.0774    1

## 7 · Alimentação cruzada

Um organismo secreta e outro capta, na mesma troca. É o que o `obj3` mediu por outra via: MIP de 1 e
zero dependências confirmadas por FBA. Se aqui aparecerem trocas com fluxo comparável ao consumo de
açúcar, isso contradiz o `obj3` e é achado; se aparecerem três ordens de grandeza abaixo, confirma-o.

In [14]:
LIMIAR = 1e-5
d_org = d_fx[d_fx.organismo != "medium"]
cols = [c for c in d_fx.columns if c.startswith("EX_") and c.endswith("_e")]

registos = []
for (nome, etiqueta), g in d_org.groupby(["consorcio", "racio"]):
    g = g.set_index("organismo")
    for ex in cols:
        v = g[ex].astype(float)
        dadores = v[v > LIMIAR]
        receptores = v[v < -LIMIAR]
        if dadores.empty or receptores.empty:
            continue
        registos.append({"consorcio": nome, "racio": etiqueta,
                         "metabolito": ex[3:-2], "troca": ex,
                         "de": ", ".join(dadores.index),
                         "para": ", ".join(receptores.index),
                         "fluxo": round(float(dadores.sum()), 6)})

d_cf = pd.DataFrame(registos)
if d_cf.empty:
    print("nenhuma troca entre organismos acima de", LIMIAR)
else:
    d_cf.to_csv(OUT / "cross_feeding.csv", index=False)
    print(f"{len(d_cf)} trocas entre organismos\n")
    print(d_cf.sort_values("fluxo", ascending=False).head(25).to_string(index=False))
    print("\npara comparacao, o consumo de hexose anda em",
          round(d_sec.hexose.median(), 2), "mmol/gDW/h")

6628 trocas entre organismos

  consorcio               racio metabolito      troca         de   para      fluxo
Sc+Td+Mp+Lt 0.95/0.02/0.02/0.02          h     EX_h_e Lt, Mp, Td     Sc 768.061436
Sc+Td+Mp+Lt 0.90/0.03/0.03/0.03          h     EX_h_e Lt, Mp, Td     Sc 759.767229
   Sc+Td+Lt      0.95/0.03/0.03          h     EX_h_e     Lt, Td     Sc 744.450161
   Sc+Td+Lt      0.90/0.05/0.05          h     EX_h_e     Lt, Td     Sc 602.250890
   Sc+Mp+Lt      0.90/0.05/0.05          h     EX_h_e     Lt, Mp     Sc 513.536332
Sc+Td+Mp+Lt 0.75/0.08/0.08/0.08          h     EX_h_e Lt, Mp, Td     Sc 443.737206
Sc+Td+Mp+Lt 0.95/0.02/0.02/0.02       malL  EX_malL_e Lt, Mp, Td     Sc 436.614682
      Sc+Lt           0.95/0.05          h     EX_h_e         Lt     Sc 433.783449
Sc+Td+Mp+Lt 0.90/0.03/0.03/0.03       malL  EX_malL_e Lt, Mp, Td     Sc 429.538487
Sc+Td+Mp+Lt 0.02/0.02/0.95/0.02          h     EX_h_e     Lt, Td Mp, Sc 414.623133
Sc+Td+Mp+Lt 0.75/0.08/0.08/0.08       malL  EX_malL_e Lt,

## 8 · Solo contra consórcio

Delta face ao µ solo do `obj2`, no mesmo meio.

**Como ler.** O `cooperative_tradeoff` impõe equidade entre membros, por isso taxas que escalam
proporcionalmente com o rácio são propriedade do objetivo e não biologia. O sinal interpretável são os
**desvios assimétricos** face a esse padrão proporcional, não o valor do delta.

In [15]:
linhas = []
for _, r in ok.iterrows():
    for cod in CONSORCIOS[r.consorcio]:
        mu = r.get(f"mu_{cod}")
        if mu is None or pd.isna(mu):
            continue
        solo = MU_OBJ2[cod]
        linhas.append({"consorcio": r.consorcio, "racio": r.racio, "organismo": cod,
                       "X": r.get(f"X_{cod}"), "mu_solo": solo,
                       "mu_consorcio": mu, "delta": round(mu - solo, 5),
                       "delta_pct": round((mu - solo) / solo * 100, 1)})

d_solo = pd.DataFrame(linhas)
d_solo.to_csv(OUT / "solo_vs_consorcio.csv", index=False)
print(d_solo.merge(chave, on=["consorcio", "racio"]).to_string(index=False))

  consorcio               racio organismo      X  mu_solo  mu_consorcio   delta  delta_pct
      Sc+Td           0.75/0.25        Sc 0.7500   0.1654       0.17517 0.00977        5.9
      Sc+Td           0.75/0.25        Td 0.2500   0.1835       0.18350 0.00000        0.0
      Sc+Mp           0.50/0.50        Sc 0.5000   0.1654       0.18448 0.01908       11.5
      Sc+Mp           0.50/0.50        Mp 0.5000   0.1836       0.18360 0.00000        0.0
      Sc+Lt           0.25/0.75        Sc 0.2500   0.1654       0.21282 0.04742       28.7
      Sc+Lt           0.25/0.75        Lt 0.7500   0.1835       0.18350 0.00000        0.0
      Td+Mp           0.25/0.75        Td 0.2500   0.1835       0.18350 0.00000        0.0
      Td+Mp           0.25/0.75        Mp 0.7500   0.1836       0.18360 0.00000        0.0
      Td+Lt           0.10/0.90        Td 0.1000   0.1835       0.18350 0.00000        0.0
      Td+Lt           0.10/0.90        Lt 0.9000   0.1835       0.18350 0.00000        0.0

## 9 · Sensibilidade à `fraction`

O `fraction` do `cooperative_tradeoff` é uma escolha, e tem de se saber quanto vale. Três valores num
rácio fixo. Se o rendimento em etanol mudar pouco, a escolha de 0.5 não decide a ordenação; se mudar
muito, o número reportado é sobre o parâmetro e não sobre o consórcio, e isso tem de ficar escrito.

In [16]:
linhas = []
for nome in ["Sc+Td", "Sc+Lt"]:
    membros = CONSORCIOS[nome]
    tax = taxonomia[taxonomia.id.isin(membros)].copy()
    for fr in (0.3, 0.5, 0.7):
        com = Community(tax, solver=SOLVER, progress=False)
        com.set_abundance({c: 1.0 / len(membros) for c in membros}, normalize=True)
        aplicar_meio_comunidade(com, meio)
        try:
            sol = com.cooperative_tradeoff(min_growth=piso(membros),
                                           fraction=fr, fluxes=True)
        except Exception as exc:
            linhas.append({"consorcio": nome, "fraction": fr,
                           "status": f"erro: {type(exc).__name__}"})
            continue
        linhas.append({"consorcio": nome, "fraction": fr, "status": sol.status,
                       "mu_comunidade": round(sol.growth_rate, 5),
                       **resumo_carbono(sol)})

d_sens = pd.DataFrame(linhas)
d_sens.to_csv(OUT / "sensibilidade_fraction.csv", index=False)
print(d_sens.to_string(index=False))

consorcio  fraction  status  mu_comunidade  hexose  etanol  Y_mol_mol  Y_g_g
    Sc+Td       0.3 optimal        0.18404 38.2798 57.7553     1.5088 0.3858
    Sc+Td       0.5 optimal        0.18404 38.2798 57.7553     1.5088 0.3858
    Sc+Td       0.7 optimal        0.18404 38.2798 57.7553     1.5088 0.3858
    Sc+Lt       0.3 optimal        0.18404 38.2798 57.6661     1.5064 0.3852
    Sc+Lt       0.5 optimal        0.18404 38.2798 57.6661     1.5064 0.3852
    Sc+Lt       0.7 optimal        0.18404 38.2798 57.6661     1.5064 0.3852


## 10 · Escolha para o dFBA

**Critério, fixado antes de correr.**

1. **Primário:** menor rendimento em etanol por hexose. Se dois consórcios ficarem a menos de 5 % um do
   outro, o modelo não os distingue e não se escolhe por aí.
2. **Secundário, e é o que desempata:** para onde vai o carbono que não foi para etanol. Desviar carbono
   para lactato ou glicerol, que se medem em vinho e têm limite sensorial conhecido, não é o mesmo que
   desviá-lo para um produto que ninguém quantificou.
3. **Levam-se dois ao dFBA, não um:** o que o modelo ordenar melhor, mais **Sc+Lt** como referência.
   A razão da referência é independente do modelo: a *L. thermotolerans* é a única das quatro com um
   mecanismo de redução de álcool documentado (desvio de carbono para ácido L-láctico), e a curadoria
   manteve-lhe a produção de lactato com tecto de secreção de 1.8 mmol/gDW/h pela edição NS4. É o
   critério já fixado no `obj3` §11.

**Limitações a declarar:**

- A abundância em estado estacionário **não é o rácio de inoculação**. Inocular 1:100 de *S. cerevisiae*
  pode dar 99 % de Sc às 48 h. A conversão entre os dois é o dFBA, e só ele.
- Daqui não sai título final de etanol em g/L. A FBA estática dá rendimentos instantâneos.
- Falta NGAM nos três CarveFungi. Sem manutenção não há fase estacionária, que é onde o vinho se faz.
  A edição NS6 existe no catálogo com o valor do yeast-GEM mas fica de fora por ser emprestado.
- O eixo mineral não é comparável entre as duas famílias (`ash 1 g` contra trocas separadas).
- Os µ de referência do `obj2` vieram do GLPK. Correr este notebook no mesmo solver dos modelos curados.

In [17]:
candidatos = melhor.head(3)[["consorcio", "racio", "Y_g_g", "mu_comunidade"]]
print("tres menores rendimentos em etanol:")
print(candidatos.to_string(index=False))

espalhamento = (candidatos.Y_g_g.max() - candidatos.Y_g_g.min()) / candidatos.Y_g_g.min()
print(f"\nespalhamento entre os tres: {espalhamento*100:.1f} %")
if espalhamento < 0.05:
    print("abaixo de 5 %: o modelo nao os distingue. Desempatar pelo perfil de secrecao.")

print("\nreferencia independente do modelo: Sc+Lt")
print(melhor[melhor.consorcio == "Sc+Lt"].to_string(index=False))

tres menores rendimentos em etanol:
consorcio          racio  Y_g_g  mu_comunidade
    Sc+Td      0.75/0.25 0.3165        0.17725
    Sc+Lt      0.25/0.75 0.3849        0.19083
 Sc+Mp+Lt 0.33/0.33/0.33 0.3900        0.18856

espalhamento entre os tres: 23.2 %

referencia independente do modelo: Sc+Lt
consorcio  n     racio  X_Sc  X_Td  status  mu_comunidade   mu_Sc  mu_Td  hexose  etanol  Y_mol_mol  Y_g_g  X_Mp  mu_Mp  X_Lt  mu_Lt
    Sc+Lt  2 0.25/0.75  0.25   NaN optimal        0.19083 0.21282    NaN 38.2785 57.6215     1.5053 0.3849   NaN    NaN  0.75 0.1835
